In [2]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import load_img, img_to_array

from tensorflow.data import Dataset

In [3]:
DATASET_DIR = "./Dataset"

images = []
for fn in os.listdir(DATASET_DIR):
    img = load_img(f"{DATASET_DIR}/{fn}", target_size=(64, 64))
    img = img_to_array(img)
    images.append(img)

In [4]:
images = np.asarray(images).astype(np.float32) / 255.0

total = len(images)
split = int(total * 0.8)

x_train = images[:split]
x_test = images[split:]

IMAGE_ORIGINAL_SHAPE = images.shape[1:]
IMAGE_SIZE = IMAGE_ORIGINAL_SHAPE[0] * IMAGE_ORIGINAL_SHAPE[1] * IMAGE_ORIGINAL_SHAPE[2]

EPOCHS = 5
BATCH_SIZE = 10
LATENT_DIM = 32
HIDDEN_DIM_256 = 256
HIDDEN_DIM_512 = 512
LEARNING_RATE = 0.0001

In [6]:
class VAE(Model):
    def __init__(self, h_dim_256, h_dim_512, l_dim, **kwargs):
        super(VAE, self).__init__(**kwargs)

        self.enc_de_1 = Dense(h_dim_512, activation="relu")
        self.enc_bn_1 = BatchNormalization()
        self.enc_do_1 = Dropout(0.2)
        self.enc_de_2 = Dense(h_dim_256, activation="relu")
        self.enc_bn_2 = BatchNormalization()
        self.enc_do_2 = Dropout(0.2)

        self.mu = Dense(l_dim)
        self.log_var = Dense(l_dim)

        self.dec_de_1 = Dense(h_dim_256)
        self.dec_bn_1 = BatchNormalization()
        self.dec_do_1 = Dropout(0.2)
        self.dec_de_2 = Dense(h_dim_512)
        self.dec_bn_2 = BatchNormalization()
        self.dec_de_out = Dense(IMAGE_SIZE)
        self.dec_bn_out = BatchNormalization()

    def encode(self, x):
        x = self.enc_de_1(x)
        x = self.enc_bn_1(x)
        x = self.enc_do_1(x)
        x = self.enc_de_2(x)
        x = self.enc_bn_2(x)
        x = self.enc_do_2(x)
        return self.mu(x), self.log_var(x)
    
    def reparam(self, mu, log_var):
        std = tf.exp(log_var * 0.5)
        eps = tf.random.normal(std.shape)
        return mu + std * eps
    
    def decode_logits(self, z):
        z = self.dec_de_1(z)
        z = self.dec_bn_1(z)
        z = self.dec_do_1(z)
        z = self.dec_de_2(z)
        z = self.dec_bn_2(z)
        z = self.dec_de_out(z)
        return self.dec_bn_out(z)
    
    def decode(self, z):
        return tf.nn.sigmoid(self.decode_logits(z))
    
    def call(self, input):
        mu, log_var = self.encode(input)
        z = self.reparam(mu, log_var)
        x_recon_logits = self.decode_logits(z)
        return x_recon_logits, mu, log_var

In [8]:
model = VAE(HIDDEN_DIM_256, HIDDEN_DIM_512, LATENT_DIM)
model(tf.zeros((BATCH_SIZE, IMAGE_SIZE)))
model.summary()

Model: "vae_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_7 (Dense)                 │ (10, 512)              │     6,291,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (10, 512)              │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (10, 256)              │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (10, 256)              │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (10, 32)               │         8,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (10, 32)               │         8,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (10, 256)              │         8,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (10, 256)              │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (10, 512)              │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (10, 512)              │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (10, 12288)            │     6,303,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (10, 12288)            │        49,152 │
│ (BatchNormalization)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,938,816 (49.36 MB)

 Trainable params: 12,911,168 (49.25 MB)

 Non-trainable params: 27,648 (108.00 KB)

In [9]:
dataset = Dataset.from_tensor_slices(x_train)
dataset = dataset.shuffle(BATCH_SIZE * 5).batch(BATCH_SIZE)
optimizer = Adam(LEARNING_RATE)

In [ ]:
loss_history = []
kl_div_history = []

for epoch in range(EPOCHS):
    for x in dataset:
        x = tf.reshape(x, [-1, IMAGE_SIZE])
        with tf.GradientTape() as tape:
            x_recon_logits, mu, log_var = model(x)
            recon_logits = tf.nn.sigmoid_cross_entropy_with_logits(x, x_recon_logits)
            recon_logits = tf.reduce_mean(tf.reduce)